> 在线讲义：[第 10 章 · 残差流与 induction head：模型怎么学会“照着前面抄”](https://wikieden.github.io/beyond-transformer/lessons/10-induction-heads.html) · 本书《超越 Transformer》

# 第 10 章 · 残差流与 induction head：模型怎么学会“照着前面抄”

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/10-induction-heads.html) · 仓库里 `beyond-transformer/notes/lessons/10-induction-heads.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 残差流 | 前 5 维 one-hot | `embed(SEQ)` | 最后一行 = A |
| 第二步 上一个词头 | 看 i − 1、写后 5 维 | `prev_token_head(x0)` | 每行看 [0, 0, 1, 2, 3, 4] |
| 第三步 induction head | q 前 5 维、k 后 5 维 | `induction_head(x1)` | 看位置 2，预测 B |
| 第三步 换序列 | 同上 | `run([0, 3, 1, 4, 2, 3])` | 预测 A |

第 1 层“上一个词 head”把前一个 token 写进残差流的另一块子空间，第 2 层 induction head 用“当前词”去找
“前一个词是它”的位置，把那个位置的词复制出来：[A][B] … [A] → [B]。
权重都是手搭的，演示机制；训练一个小模型、看 induction head 自己长出来留给里程碑 T-M2。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
TOK = ["<s>", "A", "B", "C", "D"]
SEQ = [0, 1, 2, 3, 4, 1]       # <s> A B C D A → 下一个应该是 B；<s> 是句首标记，位置 0 没有“上一个词”
V = len(TOK)

**函数 `softmax_rows`**：小工具。

In [ ]:
def softmax_rows(s):
    s = s - s.max(1, keepdims=True)
    e = np.exp(s)
    return e / e.sum(1, keepdims=True)

**函数 `embed`**：残差流 10 维：前 5 维 = 当前 token 的 one-hot；后 5 维先空着，留给第 1 层写“上一个 token”。

In [ ]:
def embed(seq=SEQ):
    """残差流 10 维：前 5 维 = 当前 token 的 one-hot；后 5 维先空着，留给第 1 层写“上一个 token”。"""
    x = np.zeros((len(seq), 2 * V))
    x[np.arange(len(seq)), seq] = 1
    return x

**函数 `prev_token_head`**：第 1 层：位置 i 的分数只在 j = i − 1 处高（位置信息手写），值 = 前 5 维，写进后 5 维。

In [ ]:
def prev_token_head(x, beta=10.0):
    """第 1 层：位置 i 的分数只在 j = i − 1 处高（位置信息手写），值 = 前 5 维，写进后 5 维。"""
    n = len(x)
    s = np.full((n, n), -1e9)
    for i in range(n):
        for j in range(i + 1):
            s[i, j] = beta if j == i - 1 else 0.0
    a = softmax_rows(s)
    out = np.zeros_like(x)
    out[:, V:] = a @ x[:, :V]
    return a, out

**函数 `induction_head`**：第 2 层：query = 当前 token（前 5 维），key = 每个位置的“上一个 token”（后 5 维），值 = 那个位置的当前 token。

In [ ]:
def induction_head(x, beta=10.0):
    """第 2 层：query = 当前 token（前 5 维），key = 每个位置的“上一个 token”（后 5 维），值 = 那个位置的当前 token。"""
    n = len(x)
    q, k, v = x[:, :V], x[:, V:], x[:, :V]
    s = beta * q @ k.T
    s[np.triu(np.ones((n, n), bool), 1)] = -1e9
    a = softmax_rows(s)
    return a, a @ v

**函数 `run`**：小工具。

In [ ]:
def run(seq=SEQ):
    x0 = embed(seq)
    a1, w1 = prev_token_head(x0)
    x1 = x0 + w1                    # 残差流：加进去，不覆盖
    a2, w2 = induction_head(x1)
    return x0, a1, x1, a2, w2

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 残差流：每层往同一条主干上加东西")
x0, a1, x1, a2, w2 = run()
print(f"  序列 {' '.join(TOK[t] for t in SEQ)}；x0 最后一行 {x0[-1].astype(int).tolist()}（前 5 维 = A）")
print("\n第二步 第 1 层：上一个词 head")
print(f"  注意力（每行最大的位置）：{a1.argmax(1).tolist()}，第 5 行权重 {np.round(a1[5], 4).tolist()}")
print(f"  x1 = x0 + 写入：位置 2（B）的残差流 {np.round(x1[2], 3).tolist()}（后 5 维 ≈ A：它前一个是 A）")

In [ ]:
print("\n第三步 第 2 层：induction head")
print(f"  最后一个位置（A）的注意力 {np.round(a2[-1], 4).tolist()}：盯住位置 2（前一个词是 A 的那个位置）")
print(f"  复制出来的向量 {np.round(w2[-1], 3).tolist()} → 预测 {TOK[int(w2[-1].argmax())]}")

In [ ]:
print("\n第四步 换一个序列：<s> C A D B C")
_, _, _, a2b, w2b = run([0, 3, 1, 4, 2, 3])
print(f"  最后一个 C 的注意力 {np.round(a2b[-1], 4).tolist()} → 预测 {TOK[int(w2b[-1].argmax())]}")

## 3. 代码里才看得到的

- **权重是手搭的**：两个头的“读哪块、写哪块”直接用切片写死，相当于 W_Q、W_K、W_V、W_O 是选取子空间的 0/1 矩阵。真实模型要从数据里学出这样的矩阵，而且子空间不会这么整齐。
- **位置信息是手写的**：上一个词头的“只看 i − 1”直接写在分数里；真实模型要靠位置编码（RoPE 等）学会。在自己的模型上训练、看它们长出来留给里程碑 T-M2。

## 4. 自己改一改

- 把 `induction_head` 的 beta 改成 1，注意力还那么集中吗？预测还对吗？
- 去掉第 1 层（x1 = x0），induction head 的 key 全是 0，注意力会变成什么样？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 残差流是共享白板，每层只加不擦；写进不同子空间，后面的头各读各的。
> 第一头看前一位，把“我前面是谁”贴胸前。
> 第二头拿着自己去比纸条，前面是我的那位，把他报出来。
> A B 再见 A，就猜 B；学的是算法，不是词对。
> 两层才搭得出，长出来那刻 loss 鼓个包，上下文学习跟着跳。